In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

# Kết nối DB
conn = sqlite3.connect("student.db")

# Tạo bảng student
student_data = {
    "student_id": [1,2,3,4,5,6,7,8,9,10],
    "name": [
        "Nguyen Minh Hoang", "Tran Thi Lan", "Pham Van Nam", "Le Thanh Huyen",
        "Vu Quoc Anh", "Dang Thuy Linh", "Bui Tien Dung", "Ho Ngoc Mai",
        "Duong Huu Phuc", "Cao Thi Hanh"
    ],
    "class": [
        "May Tinh","Kinh Te","Toan Tin","Toan Tin","May Tinh",
        "May Tinh","Kinh Te","Toan Tin","Kinh Te","May Tinh"
    ],
    "course_id": [12,34,np.nan,20,24,24,34,20,np.nan,np.nan],
    "score": [6.7,9.2,7.9,7.2,8.0,5.5,9.2,8.8,7.2,7.0]
}

df_student = pd.DataFrame(student_data)


# Tạo bảng course

course_data = {
    "id": [12,34,26],
    "course_name": ["Giai tich", "Thong ke", "Tin hoc"]
}

df_course = pd.DataFrame(course_data)

# Lưu vào SQLite
df_student.to_sql("student", conn, if_exists="replace", index=False)
df_course.to_sql("course", conn, if_exists="replace", index=False)

3

### 1. Kết nối

#### 1.1. INNER JOIN

In [ ]:
inner_join = pd.read_sql("""
SELECT s.*, c.course_name
FROM student s
INNER JOIN course c ON s.course_id = c.id
""", conn)

inner_join

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke


#### 1.2. LEFT JOIN

In [ ]:
left_join = pd.read_sql("""
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn)

left_join

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


#### 1.3. RIGHT JOIN

In [ ]:
left_join = pd.read_sql("""
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id
""", conn)

left_join

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


#### 1.4. FULL OUTER JOIN

In [ ]:
full_join = pd.read_sql("""
SELECT s.*, c.course_name
FROM student s
LEFT JOIN course c ON s.course_id = c.id

UNION

SELECT s.*, c.course_name
FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)

full_join

,student_id,name,class,course_id,score,course_name
0,NaN,None,None,NaN,NaN,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,None


#### 1.5. Làm sạch dữ liệu

In [ ]:
df = left_join.copy()

# Thay NULL course -> "Unknown"
df["course_name"] = df["course_name"].fillna("Unknown")

# Loại bỏ score NULL
df = df.dropna(subset=["score"])

### 2. Thống kê

#### 2.1. Tổng số sinh viên, điểm TB từng lớp

In [ ]:
class_stats = df.groupby("class").agg(
    total_students=("student_id", "count"),
    avg_score=("score", "mean")
).reset_index()

class_stats

,class,total_students,avg_score
0,Kinh Te,3,8.533333
1,May Tinh,4,6.800000
2,Toan Tin,3,7.966667


#### 2.2. Tổng số sinh viên, điểm TB từng môn

In [ ]:
course_stats = df.groupby("course_name").agg(
    total_students=("student_id", "count"),
    avg_score=("score", "mean")
).reset_index()

course_stats

,course_name,total_students,avg_score
0,Giai tich,1,6.700000
1,Thong ke,2,9.200000
2,Unknown,7,7.371429


#### 2.3. Phân loại sinh viên

In [ ]:
def classify(score):
    if score >= 9:
        return "Xuat sac"
    elif score >= 6:
        return "Tot"
    else:
        return "Kem"

df["rank"] = df["score"].apply(classify)

rank_stats = df["rank"].value_counts().reset_index()
rank_stats.columns = ["rank", "count"]

rank_stats

,rank,count
0,Tot,7
1,Xuat sac,2
2,Kem,1


## 3. Xếp hạng

#### 3.1. Theo điểm tổng

In [ ]:
df["rank_overall"] = df["score"].rank(ascending=False, method="dense")

#### 3.2. Theo lớp

In [ ]:
df["rank_class"] = df.groupby("class")["score"]\
                     .rank(ascending=False, method="dense")

#### 3.3. Theo môn

In [ ]:
df["rank_course"] = df.groupby("course_name")["score"]\
                      .rank(ascending=False, method="dense")

### 3.4. Top 3

In [ ]:
top3_overall = df.nsmallest(3, "rank_overall")
top3_class = df[df["rank_class"] <= 3]
top3_course = df[df["rank_course"] <= 3]

## 4. Graduation Date + Tính thời gian

In [ ]:
# Thêm cột graduation_date
df["graduation_date"] = pd.to_datetime([
    "2024-06-01","2024-07-01","2023-12-01","2024-05-15",
    "2024-08-01","2023-11-01","2024-06-20","2024-09-01",
    "2024-04-01","2024-03-01"
])

# Tính số ngày từ hiện tại
today = pd.Timestamp.today()

df["days_to_graduate"] = (df["graduation_date"] - today).dt.days